# Phutball Transformer - GPU/TPU Training

Runtime → A100 GPU or TPU v6e → Run all

In [ ]:
# Detect environment
import os
IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ or 'google.colab' in str(globals())
print(f"Running in Colab: {IN_COLAB}")

In [ ]:
# Install dependencies
if IN_COLAB:
    import subprocess
    # Check if TPU is available
    try:
        import jax
        if 'TPU' in str(jax.devices()):
            !pip install -q jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
        else:
            raise Exception("No TPU")
    except:
        # GPU or CPU - install CUDA version
        !pip install -q jax[cuda12_pip] -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
    !pip install -q flax optax wandb mctx

In [ ]:
# Verify devices
import jax
import jax.numpy as jnp

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")
print(f"Device count: {jax.device_count()}")

DEVICE_COUNT = jax.device_count()
DEVICE_TYPE = str(jax.devices()[0]).split(':')[0] if jax.devices() else 'cpu'
print(f"\nUsing {DEVICE_COUNT}x {DEVICE_TYPE}")

In [ ]:
# Mount Google Drive for checkpoints
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_BASE = "/content/drive/MyDrive/phutball_checkpoints/transformer_15x11"
else:
    CHECKPOINT_BASE = "./checkpoints_transformer_15x11"

os.makedirs(CHECKPOINT_BASE, exist_ok=True)
print(f"Checkpoints: {CHECKPOINT_BASE}")

In [ ]:
# Wandb login (optional)
USE_WANDB = True

if USE_WANDB:
    import wandb
    wandb.login()

In [ ]:
# Clone/update repo
REPO_URL = "https://github.com/echoname6/phutball-jax.git"
REPO_DIR = "/content/phutball-jax" if IN_COLAB else "./phutball-jax"

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -3

In [ ]:
# === TRAINING CONFIG ===

# Board size (15x11 - larger board with END_LO=-2 fix for directional learning)
ROWS = 15
COLS = 11

# Transformer architecture
D_MODEL = 128       # Transformer d_model (config.num_channels)
N_LAYERS = 6        # Transformer layers (config.num_res_blocks)
POS_ENCODING = "goal_distance"  # "normalized" or "goal_distance"

# Self-play (64 for A100 40GB, increase to 128 for 80GB)
BATCH_SIZE_GAMES = 64
NUM_SIMULATIONS = 32
GAMES_PER_ITER = 256

# Training
BATCH_SIZE_TRAIN = 256
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
TRAIN_STEPS_PER_ITER = 200
BUFFER_SIZE = 300_000
MIN_BUFFER_SIZE = 500

# Temperature schedule
TEMP_THRESHOLD = 30
TEMP_FINAL = 0.1

# Curriculum
CURRICULUM_ENABLED = False
CURRICULUM_INITIAL_RATIO = 0.5
CURRICULUM_FINAL_RATIO = 0.0
CURRICULUM_DECAY_ITERS = 100

# League play (play vs mixture of past checkpoints + random)
LEAGUE_ENABLED = True
LEAGUE_POOL_SIZE = 10          # Keep last N checkpoints in pool
LEAGUE_OPPONENT_RATIO = 0.3    # 30% of games vs league opponents
LEAGUE_SAVE_EVERY = 5          # Save to pool every N iterations
LEAGUE_RANDOM_RATIO = 0.1      # 10% of games vs pure random

# Iterations
NUM_ITERATIONS = 500
CHECKPOINT_EVERY = 10

# Eval (enabled for GPU - was disabled on TPU v6e due to XLA stack overflow)
EVAL_VS_RANDOM_GAMES = 50

# Notifications (ntfy.sh - install app and subscribe to topic)
NTFY_TOPIC = "phutball-transformer"  # Set to None to disable
HEARTBEAT_MINUTES = 30

WANDB_PROJECT = "phutball-transformer"

print(f"Board: {ROWS}x{COLS}")
print(f"Transformer: d_model={D_MODEL}, n_layers={N_LAYERS}, pos_encoding={POS_ENCODING}")
print(f"League play: {LEAGUE_ENABLED} (pool_size={LEAGUE_POOL_SIZE}, opp_ratio={LEAGUE_OPPONENT_RATIO})")
print(f"Training: {NUM_ITERATIONS} iterations")
print(f"Heartbeat: every {HEARTBEAT_MINUTES} min to ntfy.sh/{NTFY_TOPIC}")

In [ ]:
# Imports
import numpy as np
import time

from phutball_env_jax import (
    PhutballState, EnvConfig, reset, step, get_legal_actions,
    state_to_network_input, render_board
)
from network import PhutballTransformer, create_transformer_network
from train_batched import TransformerTrainer, TrainConfig

print("Imports OK")

In [ ]:
# Create config
config = TrainConfig(
    rows=ROWS,
    cols=COLS,

    # Transformer uses these as d_model and n_layers
    num_channels=D_MODEL,
    num_res_blocks=N_LAYERS,
    pos_encoding=POS_ENCODING,

    # Self-play
    batch_size_games=BATCH_SIZE_GAMES,
    num_simulations=NUM_SIMULATIONS,
    games_per_iteration=GAMES_PER_ITER,
    temp_threshold=TEMP_THRESHOLD,
    temp_final=TEMP_FINAL,

    # Training
    batch_size_train=BATCH_SIZE_TRAIN,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    train_steps_per_iteration=TRAIN_STEPS_PER_ITER,
    buffer_size=BUFFER_SIZE,
    min_buffer_size=MIN_BUFFER_SIZE,

    # Curriculum
    curriculum_enabled=CURRICULUM_ENABLED,
    curriculum_initial_ratio=CURRICULUM_INITIAL_RATIO,
    curriculum_final_ratio=CURRICULUM_FINAL_RATIO,
    curriculum_decay_iterations=CURRICULUM_DECAY_ITERS,

    # League play
    league_enabled=LEAGUE_ENABLED,
    league_pool_size=LEAGUE_POOL_SIZE,
    league_opponent_ratio=LEAGUE_OPPONENT_RATIO,
    league_save_every=LEAGUE_SAVE_EVERY,
    league_random_ratio=LEAGUE_RANDOM_RATIO,

    # Iterations
    num_iterations=NUM_ITERATIONS,

    # Checkpointing
    checkpoint_dir=CHECKPOINT_BASE,
    checkpoint_every=CHECKPOINT_EVERY,

    # Eval - enabled for GPU (was disabled on TPU v6e due to XLA issues)
    eval_enable=True,
    eval_vs_random_games=EVAL_VS_RANDOM_GAMES,

    # Notifications
    ntfy_topic=NTFY_TOPIC,
    heartbeat_minutes=HEARTBEAT_MINUTES,

    # Wandb
    use_wandb=USE_WANDB,
    wandb_project=WANDB_PROJECT,
    wandb_run_name=f"transformer_{ROWS}x{COLS}_d{D_MODEL}_L{N_LAYERS}_league",
)

print("Config created")

In [ ]:
# Create trainer
trainer = TransformerTrainer(config)

# Count parameters
param_count = sum(x.size for x in jax.tree_util.tree_leaves(trainer.params))
print(f"Transformer parameters: {param_count:,}")

In [ ]:
# Train!
trainer.train()

## Evaluation

In [ ]:
# Evaluate vs random
if trainer:
    print("\nFinal evaluation vs random:")
    win_rate, stats = trainer.evaluate_vs_random_batched()
    print(f"Win rate: {win_rate:.1%}")

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

if trainer and trainer.metrics_history:
    iters = [m['iteration'] for m in trainer.metrics_history]
    p_loss = [m['policy_loss'] for m in trainer.metrics_history]
    v_loss = [m['value_loss'] for m in trainer.metrics_history]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(iters, p_loss)
    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('Policy Loss')
    axes[0].set_title('Policy Loss')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(iters, v_loss)
    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('Value Loss')
    axes[1].set_title('Value Loss')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No metrics to plot")

In [ ]:
# List checkpoints
import glob

print("Available checkpoints:")
ckpts = sorted(glob.glob(os.path.join(CHECKPOINT_BASE, "*.pkl")))
print(f"Total: {len(ckpts)}")
for c in ckpts[-5:]:
    print(f"  - {os.path.basename(c)}")

In [ ]:
# Delete old checkpoints (add # before the for loop to skip deletion)
import glob

ckpts = sorted(glob.glob(os.path.join(CHECKPOINT_BASE, "*.pkl")))
print(f"Found {len(ckpts)} checkpoints:")
for c in ckpts:
    print(f"  {os.path.basename(c)}")

for c in ckpts:
    os.remove(c)
print(f"\nDeleted {len(ckpts)} checkpoints")